# Filesystems

> For filesystems that store video, text, and more

In [ ]:
# | default_exp filesystems


In [ ]:
# | export
from typing import Any
from typing_extensions import Annotated
from pydantic import BaseModel, StringConstraints
from dataclasses import dataclass, field
from enum import Enum
from abc import ABC, abstractmethod

In [ ]:
# | hide
from nbdev.showdoc import *

In [ ]:
# | export
class File(BaseModel):
    uri: str


class AudioFile(File):
    start: int
    end: int
    file: str


class VideoFile(File):
    start: int
    end: int
    file: str


class TextFile(File):
    pass

In [ ]:
# | export


class User(BaseModel):
    id: Annotated[str, StringConstraints(min_length=1)]


class FileSystemType(Enum):
    LOCAL = "local"
    R2 = "r2"


class FilePathType(Enum):
    """Files can come in as Audio, Video, Text, or PDF.
    The original file should be saved in user_id/originals.
    Transformed files should be saved in user_id/audio, user_id/video, user_id/text, user_id/pdf
    Relationships back to original file should be saved in the db, but that's not handled by filesystem APIs"""

    USER_ID_BASE = "user_id"
    USER_ID_BASE_AUDIO = "user_id_audio"
    USER_ID_BASE_VIDEO = "user_id_video"
    USER_ID_BASE_TEXT = "user_id_text"
    USER_ID_BASE_PDF = "user_id_pdf"


@dataclass
class AbstractFileSystem(ABC):
    base_path: str = field(default="")

    @abstractmethod
    def build_user_path(self, user: User, file_path_type: FilePathType) -> str:
        """Build the file path according to the inputted objects and rules.
        Does not create resources, just a string formatter"""
        raise NotImplementedError("This method should be implemented by subclasses")

    @abstractmethod
    def create_folder(self, path: str) -> str:
        """Creates folders at the specified location. Doesn't raise errors if locaqtion exists"""
        raise NotImplementedError("This method should be implemented by subclasses")

    def sanitize_characters(self, text: str) -> str:
        """Replace non-ASCII characters with their Unicode codepoint representations."""
        return text.encode("unicode_escape").decode().replace("\\u", "u").replace("\\", "_").replace('\\U', 'U')

    def sanitize_filename(self, filename: str) -> str:
        """Use as an add-on to sanitize characters. Replace non-ASCII characters with their Unicode codepoint representations."""
        return filename.replace("/", "_").replace("\\", "_")

    @abstractmethod
    def validate_access(self, user: User, path: str) -> bool:
        """Check if the user has access to the specified location"""
        raise NotImplementedError("This method should be implemented by subclasses")

    @abstractmethod
    def check_exists(self, path: str) -> bool:
        """Check if the file exists at the specified path."""
        raise NotImplementedError("This method should be implemented by subclasses")

    @abstractmethod
    def create_file(self, path: str, content: bytes, name: str, authorized: bool = False) -> File:
        """Creates files at specified path. Throws an error if the location doesn't exist.
        Does NOT check permissions — pass in authorized to allow access"""
        raise NotImplementedError("This method should be implemented by subclasses")
    
    @abstractmethod
    def read_file(self, path: str) -> File:
        raise NotImplementedError("This method should be implemented by subclasses")

    @abstractmethod
    def update_file(self, path: str, content: bytes) -> File:
        raise NotImplementedError("This method should be implemented by subclasses")

    @abstractmethod
    def delete_file(self, path: str) -> File:
        raise NotImplementedError("This method should be implemented by subclasses")

In [ ]:
# | export
import os
import tempfile


@dataclass
class LocalFileSystem(AbstractFileSystem):
    base_path: str = field(
        default=tempfile.mkdtemp(),
        metadata={"help": "The base path for the local file system."},
    )

    def build_user_path(
        self: AbstractFileSystem, user: User, file_path_type: FilePathType
    ) -> str:
        if file_path_type == FilePathType.USER_ID_BASE:
            return f"{self.base_path}/{user.id}/originals"
        elif file_path_type == FilePathType.USER_ID_BASE_AUDIO:
            return f"{self.base_path}/{user.id}/audio"
        elif file_path_type == FilePathType.USER_ID_BASE_VIDEO:
            return f"{self.base_path}/{user.id}/video"
        elif file_path_type == FilePathType.USER_ID_BASE_TEXT:
            return f"{self.base_path}/{user.id}/text"
        elif file_path_type == FilePathType.USER_ID_BASE_PDF:
            return f"{self.base_path}/{user.id}/pdf"
        else:
            raise ValueError(f"Invalid file path type: {file_path_type}")

    def create_folder(self: AbstractFileSystem, path: str) -> str:
        path = self.sanitize_characters(path)
        os.makedirs(path, exist_ok=True)
        return path

    def validate_access(self: AbstractFileSystem, user: User, path: str) -> bool:
        # Simplified access validation: checks if path starts with user's directory
        return str(user.id) in path

    def check_exists(self: AbstractFileSystem, path: str) -> bool:
        return os.path.exists(path)

    def create_file(self: AbstractFileSystem, path: str, content: bytes, name: str, authorized: bool = False) -> File:
        if not authorized:
            raise FileNotFoundError("Access denied")
        if not self.check_exists(os.path.dirname(path)):
            raise FileNotFoundError("Location doesn't exist")
        sanitized_name = self.sanitize_characters(self.sanitize_filename(name))
        with open(path + "/" + sanitized_name, "wb") as f:
            f.write(content)
        return File(uri=path + "/" + sanitized_name)

    def read_file(self: AbstractFileSystem, path: str) -> File:
        if not os.path.isfile(path):
            raise FileNotFoundError("File does not exist")
        return File(uri=path)

    def update_file(self: AbstractFileSystem, path: str, content: bytes) -> File:
        if not os.path.isfile(path):
            raise FileNotFoundError("File does not exist")
        return File(uri=path)

    def delete_file(self: AbstractFileSystem, path: str) -> File:
        if not os.path.isfile(path):
            raise FileNotFoundError("File does not exist")
        os.remove(path)
        return File(uri=path)

Below is the canonical FileSystem test. If a state machine passes the test below, it functions as intended.

Use it as a reference for how to use the filesystem apis and the order in which to use it

In [ ]:
from hypothesis import strategies as st
from hypothesis.stateful import (
    RuleBasedStateMachine,
    rule,
    precondition,
    Bundle,
    consumes,
)
from product_horse.core import run_test_case_with_pdb
# import tempfile
import shutil
# from pathlib import Path


class FileSystemStateMachine(RuleBasedStateMachine):
    def __init__(self):
        super().__init__()
        self.file_system = LocalFileSystem()
        self.file_count = 0
        self.user_count = 0

    def teardown(self):
        assert 'tmp' in self.file_system.base_path
        try:
            shutil.rmtree(self.file_system.base_path)
        except FileNotFoundError:
            pass

    files = Bundle("files")
    user_paths = Bundle("user_paths")
    file_paths = Bundle("file_paths")

    @rule(
        target=user_paths,
        user=st.builds(User),
        file_path_type=st.sampled_from(FilePathType),
    )
    def build_user_path(self, user: User, file_path_type: FilePathType) -> str:
        """Format the correct user path for the file system and file type, check access"""
        self.user_count += 1
        path = self.file_system.build_user_path(user, file_path_type)
        assert self.file_system.base_path in path
        assert self.file_system.validate_access(user, path)
        return path

    @rule(target=file_paths, path=user_paths)
    @precondition(lambda self: self.user_count > 0)
    def create_folder(self, path: str) -> str:
        """Create a folder in the file system for the user, based on user_path.
        Returns no exception if path exists already."""
        created_path = self.file_system.create_folder(path)
        assert os.path.exists(created_path)
        return  created_path

    @rule(target=files, path=file_paths, content=st.binary(min_size=1), filename=st.text(min_size=1))
    def create_file(self, path: str, content: bytes, filename: str) -> File:
        """Create a file in the file system."""
        file = self.file_system.create_file(path, content, name=filename, authorized=True)
        assert self.file_system.check_exists(file.uri)
        self.file_count += 1
        return file

    @rule(file=consumes(files))
    @precondition(lambda self: self.file_count > 0)
    def delete_file(self, file: File):
        self.file_system.delete_file(file.uri)
        self.file_count -= 1

    @rule(file=files)
    @precondition(lambda self: self.file_count > 0)
    def read_file(self, file: File):
        assert self.file_system.read_file(file.uri).uri == file.uri

    @rule(file=files, content=st.binary(min_size=1))
    @precondition(lambda self: self.file_count > 0)
    def update_file(self, file: File, content: bytes):
        self.file_system.update_file(file.uri, content)
        assert self.file_system.check_exists(file.uri)


TestFileSystem = FileSystemStateMachine.TestCase
run_test_case_with_pdb(TestFileSystem, pdb_flag=False)

runTest (hypothesis.stateful.FileSystemStateMachine.TestCase.runTest) ... ok

----------------------------------------------------------------------
Ran 1 test in 48.497s

OK


In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()  # type: ignore